# 10 · Build a Llama-style decoder block and count its parameters

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/architecture/10_llama_block.ipynb) &nbsp; 📖 Concept deck: [Putting it together](https://nilesh-salpe.github.io/visual-ai/architecture/11-putting-it-together/) &nbsp; ⏱️ Suggested time: **50 min**

**You will build:** a complete modern decoder block (pre-norm RMSNorm, GQA with RoPE, SwiGLU), a parameter counter that reproduces Llama 2 7B and Llama 3 8B exactly, and a check that the block is causal.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def _rms_ref(x, g, eps=1e-6):
    return x / np.sqrt((x * x).mean(-1, keepdims=True) + eps) * g

def _ln_ref(x, g, b, eps=1e-6):
    return (x - x.mean(-1, keepdims=True)) / np.sqrt(x.var(-1, keepdims=True) + eps) * g + b

def _rope_ref(x, pos, base=10000.0):
    """Interleaved-pair RoPE as in RoFormer: pairs (x0,x1), (x2,x3), ... rotate by pos * theta_i."""
    T, d = x.shape
    theta = base ** (-np.arange(0, d, 2) / d)            # [d/2]
    ang = np.asarray(pos, float)[:, None] * theta[None]  # [T, d/2]
    c, s = np.cos(ang), np.sin(ang)
    x0, x1 = x[:, 0::2], x[:, 1::2]
    out = np.empty_like(x, dtype=float)
    out[:, 0::2] = x0 * c - x1 * s
    out[:, 1::2] = x0 * s + x1 * c
    return out

def _silu(x):
    return x / (1 + np.exp(-x))

def _swiglu_ref(x, W_gate, W_up, W_down):
    return (_silu(x @ W_gate) * (x @ W_up)) @ W_down

def init_block(d=64, n_heads=8, n_kv=2, hidden=176, seed=0):
    r = np.random.default_rng(seed); dh = d // n_heads
    n = lambda *s: r.normal(0, s[0] ** -.5, s)
    return dict(n_heads=n_heads, n_kv=n_kv, g_attn=np.ones(d), g_ffn=np.ones(d),
                Wq=n(d, n_heads * dh), Wk=n(d, n_kv * dh), Wv=n(d, n_kv * dh), Wo=n(n_heads * dh, d),
                W_gate=n(d, hidden), W_up=n(d, hidden), W_down=n(hidden, d))
B = init_block()

## Task 1: Grouped-query attention with RoPE

1. Project: $q = xW_q$ → [T, n_heads, dh]; $k = xW_k$ and $v = xW_v$ → [T, n_kv, dh].
2. Apply RoPE (`_rope_ref(x, pos)`, per head, positions 0..T-1) to **q and k only**.
3. Query head $h$ uses KV head $h \div (n_{heads}/n_{kv})$, so consecutive query heads share a KV head.
4. Causal softmax of $qk^\top/\sqrt{dh}$, multiply by $v$, concatenate heads, then $\cdot W_o$.

In [ ]:
def gqa_rope_attention(x, B):
    """x: [T, d] (already normed). Returns [T, d]."""
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_gqa_rope_attention(f):
    x = np.random.default_rng(1).normal(size=(9, 64)); y = f(x, B)
    assert y.shape == (9, 64)
    T, H, G = 9, 8, 2; dh = 8; pos = np.arange(T)
    q = (x @ B['Wq']).reshape(T, H, dh); k = (x @ B['Wk']).reshape(T, G, dh); v = (x @ B['Wv']).reshape(T, G, dh)
    outs = []
    for h in range(H):
        g = h // (H // G); s = _rope_ref(q[:, h], pos) @ _rope_ref(k[:, g], pos).T / np.sqrt(dh)
        outs.append(softmax(np.where(np.tril(np.ones((T, T), bool)), s, -np.inf)) @ v[:, g])
    assert np.allclose(y, np.concatenate(outs, 1) @ B['Wo'], atol=1e-10), "values differ: check RoPE on q and k, the head-to-group mapping h // (H // G), and the causal mask"

check("gqa_rope_attention", _test_gqa_rope_attention, gqa_rope_attention)

## Task 2: The full block

$$h = x + \mathrm{Attn}(\mathrm{RMSNorm}(x)), \qquad \text{out} = h + \mathrm{SwiGLU}(\mathrm{RMSNorm}(h))$$
Use `_rms_ref(x, g)` with `B['g_attn']` and `B['g_ffn']`, your `gqa_rope_attention` (or the `_solution`), and `_swiglu_ref`. No biases anywhere. The test also checks that changing the last token leaves earlier outputs untouched.

In [ ]:
def llama_block(x, B):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_llama_block(f):
    x = np.random.default_rng(2).normal(size=(7, 64)); y = f(x, B)
    attn = globals().get('gqa_rope_attention_solution') or _ref_attn
    h = x + attn(_rms_ref(x, B['g_attn']), B)
    ref = h + _swiglu_ref(_rms_ref(h, B['g_ffn']), B['W_gate'], B['W_up'], B['W_down'])
    assert y.shape == x.shape
    assert np.allclose(y, ref, atol=1e-10), "values differ: pre-norm RMSNorm -> attention -> residual, then pre-norm RMSNorm -> SwiGLU -> residual"
    x2 = x.copy(); x2[-1] += 3
    assert np.allclose(f(x2, B)[:-1], y[:-1]), "the block must be causal"

def _ref_attn(x, B):
    T, d = x.shape; H, G = B['n_heads'], B['n_kv']; dh = B['Wq'].shape[1] // H; pos = np.arange(T)
    q = (x @ B['Wq']).reshape(T, H, dh); k = (x @ B['Wk']).reshape(T, G, dh); v = (x @ B['Wv']).reshape(T, G, dh)
    outs = [softmax(np.where(np.tril(np.ones((T, T), bool)), _rope_ref(q[:, h], pos) @ _rope_ref(k[:, h // (H // G)], pos).T / np.sqrt(dh), -np.inf)) @ v[:, h // (H // G)] for h in range(H)]
    return np.concatenate(outs, axis=1) @ B['Wo']

check("llama_block", _test_llama_block, llama_block)

## Task 3: Count a real model's parameters

Given a config, return total parameters:
* embeddings: `vocab × d`; output head: another `vocab × d` unless `tied`
* per layer: $W_q$ and $W_o$ ($d\times d$ each); $W_k$ and $W_v$ ($d \times n_{kv}\cdot d_h$ each, $d_h = d/n_{heads}$); SwiGLU ($3\cdot d\cdot h$); two RMSNorm gains ($2d$)
* one final RMSNorm ($d$)

Expected: **Llama 2 7B = 6,738,415,616** and **Llama 3 8B = 8,030,261,248**.

In [ ]:
def count_params(vocab, d, n_layers, n_heads, n_kv, hidden, tied=False):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_count_params(f):
    l2 = f(32000, 4096, 32, 32, 32, 11008)
    assert l2 == 6_738_415_616, f"Llama 2 7B: got {l2:,}, expected 6,738,415,616"
    l3 = f(128256, 4096, 32, 32, 8, 14336)
    assert l3 == 8_030_261_248, f"Llama 3 8B: got {l3:,}, expected 8,030,261,248"
    assert f(100, 8, 1, 2, 2, 16, tied=True) == 100 * 8 + (4 * 64 + 3 * 8 * 16 + 16) + 8, "tied embeddings count once"

check("count_params", _test_count_params, count_params)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Grouped-query attention with RoPE
def gqa_rope_attention_solution(x, B):
    T, d = x.shape; H, G = B['n_heads'], B['n_kv']; dh = B['Wq'].shape[1] // H; pos = np.arange(T)
    q = (x @ B['Wq']).reshape(T, H, dh); k = (x @ B['Wk']).reshape(T, G, dh); v = (x @ B['Wv']).reshape(T, G, dh)
    mask = np.tril(np.ones((T, T), bool)); outs = []
    for h in range(H):
        g = h // (H // G)
        s = _rope_ref(q[:, h], pos) @ _rope_ref(k[:, g], pos).T / np.sqrt(dh)
        outs.append(softmax(np.where(mask, s, -np.inf)) @ v[:, g])
    return np.concatenate(outs, axis=1) @ B['Wo']

check("gqa_rope_attention (reference)", _test_gqa_rope_attention, gqa_rope_attention_solution)

In [ ]:
# @title Solution for Task 2: The full block
def llama_block_solution(x, B):
    h = x + gqa_rope_attention_solution(_rms_ref(x, B['g_attn']), B)
    return h + _swiglu_ref(_rms_ref(h, B['g_ffn']), B['W_gate'], B['W_up'], B['W_down'])

check("llama_block (reference)", _test_llama_block, llama_block_solution)

In [ ]:
# @title Solution for Task 3: Count a real model's parameters
def count_params_solution(vocab, d, n_layers, n_heads, n_kv, hidden, tied=False):
    dh = d // n_heads
    attn = 2 * d * d + 2 * d * n_kv * dh
    layer = attn + 3 * d * hidden + 2 * d
    return vocab * d * (1 if tied else 2) + n_layers * layer + d

check("count_params (reference)", _test_count_params, count_params_solution)

---
## Experiments
**Where do the parameters go?** Break down Llama 3 8B, and see how GQA and the 128K vocabulary shifted the budget compared with Llama 2 7B.

In [ ]:
def breakdown(name, vocab, d, L, H, G, hid):
    dh = d // H; emb = 2 * vocab * d; attn = L * (2 * d * d + 2 * d * G * dh); ffn = L * 3 * d * hid
    tot = count_params_solution(vocab, d, L, H, G, hid)
    print(f"{name}: total {tot / 1e9:.2f}B | embeddings+head {emb / tot:5.1%} | attention {attn / tot:5.1%} | FFN {ffn / tot:5.1%}")
breakdown('Llama 2 7B ', 32000, 4096, 32, 32, 32, 11008)
breakdown('Llama 3 8B ', 128256, 4096, 32, 32, 8, 14336)
x = np.random.default_rng(0).normal(size=(16, 64))
y = x
for layer in range(4):
    y = llama_block_solution(y, init_block(seed=layer))
print(f"\n4 random blocks ran end to end: output shape {y.shape}, residual RMS grew from {np.sqrt((x**2).mean()):.2f} to {np.sqrt((y**2).mean()):.2f} (why a final RMSNorm is needed)")

---
## Takeaways
* A Llama block is just the 2017 block with four swaps: pre-norm, RMSNorm, RoPE inside attention, SwiGLU FFN. Llama 2 70B and Llama 3 add GQA.
* Parameter counts follow directly from the config. Being able to reproduce 8.03B from `config.json` is a good sanity check of your mental model.
* In Llama 3 8B the FFN is about 70% of all parameters, attention (thanks to GQA) about 17%, and the 128K-vocabulary embeddings plus output head about 13%.